In [ ]:
import numpy as np
import pandas as pd

from datetime import datetime
from autogluon.tabular import TabularDataset, TabularPredictor
# from extinction import fitzpatrick99
from pathlib import Path
from typing import Literal, Tuple, Dict, Optional, Union, Iterable, Self
from scipy.optimize import curve_fit
from scipy.stats import linregress
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion, Pipeline

c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
EPS = np.finfo(float).eps

### Data Loading

In [4]:
def load_dfs(type: Literal["train", "test"]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    log_df = __load_log_df(type)
    
    splits = sorted(log_df["split"].unique())
    flc_dfs = [__load_flc_df(type, split) for split in splits]
    flc_df = pd.concat(flc_dfs)

    return (log_df, flc_df)

def __load_log_df(type: Literal["train", "test"]) -> pd.DataFrame:
    return pd.read_csv(f"../resources/kaggle/{type}_log.csv", index_col="object_id").drop(columns=["English Translation"])

def __load_flc_df(type: Literal["train", "test"], split: str) -> pd.DataFrame:
    return pd.read_csv(f"../resources/kaggle/{split}/{type}_full_lightcurves.csv")

In [5]:
train_log_df, train_flc_df = load_dfs("train")
test_log_df, test_flc_df = load_dfs("test")

train_flc_df

,object_id,Time (MJD),Flux,Flux_err,Filter
0,Dornhoth_fervain_onodrim,63314.4662,-1.630159,0.365777,z
1,Dornhoth_fervain_onodrim,63780.9674,10.499389,0.253867,r
2,Dornhoth_fervain_onodrim,63789.7693,5.866250,1.559241,y
3,Dornhoth_fervain_onodrim,63794.1702,3.903623,0.376854,r
4,Dornhoth_fervain_onodrim,63794.1702,5.226644,0.516864,i
...,...,...,...,...,...
23514,ylf_alph_mindon,63490.0577,0.102033,0.180718,i
23515,ylf_alph_mindon,63494.7890,0.086665,0.356742,i
23516,ylf_alph_mindon,63530.2733,-0.284907,0.235458,r
23517,ylf_alph_mindon,63324.4641,3.270126,1.897147,i


### Preprocessing

In [6]:
def de_extinct(log_df: pd.DataFrame, flc_df: pd.DataFrame):
    EXTINCTION_COEFFS = {
        "u": 4.81,
        "g": 3.64,
        "r": 2.70,
        "i": 2.06,
        "z": 1.58,
        "y": 1.31
    }

    ebv = flc_df["object_id"].map(log_df["EBV"])
    r_λ = flc_df["Filter"].map(EXTINCTION_COEFFS)

    c_λ = np.pow(10, 0.4 * r_λ * ebv)

    flc_df["Flux"] *= c_λ
    flc_df["Flux_err"] *= c_λ


# TODO Consider extinction.fitzpatrick99

In [7]:
de_extinct(train_log_df, train_flc_df)

train_flc_df

,object_id,Time (MJD),Flux,Flux_err,Filter
0,Dornhoth_fervain_onodrim,63314.4662,-1.913154,0.429276,z
1,Dornhoth_fervain_onodrim,63780.9674,13.802700,0.333739,r
2,Dornhoth_fervain_onodrim,63789.7693,6.698855,1.780546,y
3,Dornhoth_fervain_onodrim,63794.1702,5.131778,0.495420,r
4,Dornhoth_fervain_onodrim,63794.1702,6.439659,0.636819,i
...,...,...,...,...,...
23514,ylf_alph_mindon,63490.0577,0.108832,0.192760,i
23515,ylf_alph_mindon,63494.7890,0.092440,0.380513,i
23516,ylf_alph_mindon,63530.2733,-0.310044,0.256232,r
23517,ylf_alph_mindon,63324.4641,3.488033,2.023565,i


### Feature Engineering

In [8]:
import pandas as pd
import numpy as np
from typing import List, Dict
from scipy.optimize import curve_fit

# Constants
EPS = np.finfo(float).eps
FILTERS = ["u", "g", "r", "i", "z", "y"]

class TDEFeatureEngineer:
    def __init__(self):
        # We cache the grouped object to avoid repeated expensive groupbys
        self._grouped_cache = None
        self._grouped_cache_id = None  # FIX: Initialize cache ID

    def _get_grouped(self, flc_df: pd.DataFrame):
        """Helper to cache groupby object. 
        Resets if the dataframe object id changes (simple check)."""
        if self._grouped_cache is None or self._grouped_cache_id != id(flc_df):
            self._grouped_cache = flc_df.groupby("object_id")
            self._grouped_cache_id = id(flc_df)
        return self._grouped_cache

    # ---------------------------------------------------------
    # 1. Peak Luminosity (Flux)
    # ---------------------------------------------------------
    def derive_peak_luminosity(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
        """
        Extracts the maximum flux for each band.
        """
        grouped = self._get_grouped(flc_df)
        
        # Efficient pandas way:
        max_flux = flc_df.groupby(["object_id", "Filter"])["Flux"].max().unstack(fill_value=np.nan)
        max_flux.columns = [f"max_flux_{col}" for col in max_flux.columns]
        
        return max_flux

    # ---------------------------------------------------------
    # 2. Rise Time
    # ---------------------------------------------------------
    def derive_rise_time(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculates time from first observation to peak flux per band.
        Formula: t_peak - t_first
        """
        grouped = self._get_grouped(flc_df)
        
        def calc_rise(obj_df):
            res = {}
            for band in FILTERS:
                band_data = obj_df[obj_df["Filter"] == band].copy()
                if len(band_data) == 0:
                    res[f"rise_time_{band}"] = np.nan
                    continue
                
                # FIX: Check if all flux values are NaN
                if band_data["Flux"].isna().all():
                    res[f"rise_time_{band}"] = np.nan
                    continue
                
                t_min = band_data["Time (MJD)"].min()
                # Use idxmax() to get the index, then loc to get the time
                peak_idx = band_data["Flux"].idxmax()
                
                # FIX: Check if peak_idx is valid (not NaN)
                if pd.isna(peak_idx):
                    res[f"rise_time_{band}"] = np.nan
                    continue
                    
                t_peak = band_data.loc[peak_idx, "Time (MJD)"]
                res[f"rise_time_{band}"] = np.maximum(t_peak - t_min, 0)
            return pd.Series(res)

        return grouped.apply(calc_rise)

    # ---------------------------------------------------------
    # 3. Power Law Decay (Alpha)
    # ---------------------------------------------------------
    def derive_power_law_decay(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
        """
        Fits F ~ t^-alpha for t > t_peak.
        Returns the alpha index. TDEs expect alpha ~ 5/3 (1.67).
        """
        grouped = self._get_grouped(flc_df)

        def calc_decay(obj_df):
            res = {}
            for band in FILTERS:
                band_data = obj_df[obj_df["Filter"] == band].copy()
                if len(band_data) < 5: # Need points for fit
                    res[f"decay_alpha_{band}"] = np.nan
                    continue
                
                # FIX: Reset index to avoid iloc/loc conflicts
                band_data = band_data.reset_index(drop=True)
                
                # Identify Peak
                peak_idx = band_data["Flux"].idxmax()
                t_peak = band_data.loc[peak_idx, "Time (MJD)"]
                
                # Select Decay Phase (t > t_peak)
                decay_phase = band_data[band_data["Time (MJD)"] > t_peak].copy()
                
                if len(decay_phase) < 3:
                    res[f"decay_alpha_{band}"] = np.nan
                    continue
                
                # Fit: ln(Flux) = -alpha * ln(t - t_peak) + C
                t_rel = decay_phase["Time (MJD)"].values - t_peak
                flux = decay_phase["Flux"].values
                
                # Handle numerics - filter out non-positive values
                valid_mask = (t_rel > 0.1) & (flux > EPS)
                if valid_mask.sum() < 3:
                    res[f"decay_alpha_{band}"] = np.nan
                    continue
                
                x = np.log(t_rel[valid_mask])
                y = np.log(flux[valid_mask])
                
                try:
                    # Polyfit degree 1 -> returns [slope, intercept]
                    # slope = -alpha
                    slope, _ = np.polyfit(x, y, 1)
                    res[f"decay_alpha_{band}"] = -slope
                except:
                    res[f"decay_alpha_{band}"] = np.nan
            return pd.Series(res)

        return grouped.apply(calc_decay)

    # ---------------------------------------------------------
    # 4. LC Smoothness (Chi-Squared of Power Law)
    # ---------------------------------------------------------
    def derive_lc_smoothness(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculates residuals of the power law fit.
        Small residuals = Smooth (TDE). Large residuals = Stochastic (AGN).
        """
        grouped = self._get_grouped(flc_df)

        def calc_smoothness(obj_df):
            res = {}
            for band in FILTERS:
                band_data = obj_df[obj_df["Filter"] == band].copy()
                if len(band_data) < 5: 
                    res[f"smoothness_{band}"] = np.nan
                    continue
                
                # FIX: Reset index
                band_data = band_data.reset_index(drop=True)
                
                peak_idx = band_data["Flux"].idxmax()
                t_peak = band_data.loc[peak_idx, "Time (MJD)"]
                decay_phase = band_data[band_data["Time (MJD)"] > t_peak].copy()
                
                if len(decay_phase) < 3:
                    res[f"smoothness_{band}"] = np.nan
                    continue

                t_rel = decay_phase["Time (MJD)"].values - t_peak
                flux = decay_phase["Flux"].values
                
                # FIX: Filter valid values
                valid_mask = (t_rel > 0.1) & (flux > EPS)
                if valid_mask.sum() < 3:
                    res[f"smoothness_{band}"] = np.nan
                    continue
                
                x = np.log(t_rel[valid_mask])
                y = np.log(flux[valid_mask])
                
                try:
                    slope, intercept = np.polyfit(x, y, 1)
                    
                    # Calculate predicted Flux in log space
                    y_pred = slope * x + intercept
                    
                    # Mean Squared Error in Log Space (Standardized)
                    # This represents deviation from the "ideal" power law shape
                    mse = np.mean((y - y_pred)**2)
                    res[f"smoothness_{band}"] = mse
                except:
                    res[f"smoothness_{band}"] = np.nan
            return pd.Series(res)

        return grouped.apply(calc_smoothness)

    # ---------------------------------------------------------
    # 5. Mean Blue Color (Temperature Proxy)
    # ---------------------------------------------------------
    def derive_mean_blue_color(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculates flux ratios at peak.
        Blue Color = Flux(u) / Flux(g)
        """
        # Calculate max flux per band per object
        max_flux = flc_df.groupby(["object_id", "Filter"])["Flux"].max().unstack(fill_value=np.nan)
        
        res = pd.DataFrame(index=max_flux.index)
        
        # u/g ratio (Bluest)
        if "u" in max_flux.columns and "g" in max_flux.columns:
            res["color_u_g"] = max_flux["u"] / (max_flux["g"] + EPS)
        else:
            res["color_u_g"] = np.nan
            
        # g/r ratio
        if "g" in max_flux.columns and "r" in max_flux.columns:
            res["color_g_r"] = max_flux["g"] / (max_flux["r"] + EPS)
        else:
            res["color_g_r"] = np.nan
            
        return res

    # ---------------------------------------------------------
    # 6. Color Evolution (TDE vs SNe)
    # ---------------------------------------------------------
    def derive_color_evolution(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculates change in color over time. 
        Formula: (g/r)_late - (g/r)_peak
        TDEs (Blackbody) -> ~0 change.
        SNe (Cooling) -> Decreasing ratio (Reddening).
        """
        grouped = self._get_grouped(flc_df)

        def calc_evolution(obj_df):
            # Focus on g and r bands (most reliable usually)
            g_band = obj_df[obj_df["Filter"] == "g"].copy()
            r_band = obj_df[obj_df["Filter"] == "r"].copy()
            
            if g_band.empty or r_band.empty:
                return pd.Series({"color_evol_g_r": np.nan})
            
            # Find global peak time (approximate center of event)
            # We use the peak of the brightest available band as reference
            peak_idx = obj_df["Flux"].idxmax()
            peak_time = obj_df.loc[peak_idx, "Time (MJD)"]
            
            # 1. Peak Color (Window: Peak +/- 10 days)
            mask_peak_g = (g_band["Time (MJD)"] - peak_time).abs() < 10
            mask_peak_r = (r_band["Time (MJD)"] - peak_time).abs() < 10
            
            flux_g_peak = g_band.loc[mask_peak_g, "Flux"].mean()
            flux_r_peak = r_band.loc[mask_peak_r, "Flux"].mean()
            
            if pd.isna(flux_g_peak) or pd.isna(flux_r_peak) or flux_r_peak == 0:
                return pd.Series({"color_evol_g_r": np.nan})
                
            color_peak = flux_g_peak / (flux_r_peak + EPS)
            
            # 2. Late Color (Window: Peak + [20 to 60] days)
            mask_late_g = (g_band["Time (MJD)"] - peak_time).between(20, 60)
            mask_late_r = (r_band["Time (MJD)"] - peak_time).between(20, 60)
            
            flux_g_late = g_band.loc[mask_late_g, "Flux"].mean()
            flux_r_late = r_band.loc[mask_late_r, "Flux"].mean()
            
            if pd.isna(flux_g_late) or pd.isna(flux_r_late) or flux_r_late == 0:
                # If we have no late data, evolution is unknown
                return pd.Series({"color_evol_g_r": np.nan})
                
            color_late = flux_g_late / (flux_r_late + EPS)
            
            # Evolution = Late - Peak
            return pd.Series({"color_evol_g_r": color_late - color_peak})

        return grouped.apply(calc_evolution)

    # ---------------------------------------------------------
    # MASTER UPDATE FUNCTION
    # ---------------------------------------------------------
    def generate_features(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
        """
        Orchestrates the feature generation pipeline.
        Returns a DataFrame containing all engineered features, indexed by object_id.
        """
        print("Starting Feature Engineering...")
        
        # 1. Peak Luminosity
        print("  - Deriving Peak Luminosity...")
        feat_peak = self.derive_peak_luminosity(log_df, flc_df)
        
        # 2. Rise Time
        print("  - Deriving Rise Time...")
        feat_rise = self.derive_rise_time(log_df, flc_df)
        
        # 3. Power Law Decay
        print("  - Deriving Power Law Decay...")
        feat_decay = self.derive_power_law_decay(log_df, flc_df)
        
        # 4. Smoothness
        print("  - Deriving LC Smoothness...")
        feat_smooth = self.derive_lc_smoothness(log_df, flc_df)
        
        # 5. Mean Blue Color
        print("  - Deriving Mean Blue Color...")
        feat_color_mean = self.derive_mean_blue_color(log_df, flc_df)
        
        # 6. Color Evolution
        print("  - Deriving Color Evolution...")
        feat_color_evol = self.derive_color_evolution(log_df, flc_df)
        
        # Combine all features
        print("  - Consolidating features...")
        features = pd.concat([
            feat_peak,
            feat_rise,
            feat_decay,
            feat_smooth,
            feat_color_mean,
            feat_color_evol
        ], axis=1)
        
        # Finally, join with original log info (Redshift Z, etc.)
        # We assume log_df has 'Z' and 'Z_err' which are crucial features themselves
        # FIX: Set index on log_df if needed
        if "object_id" in log_df.columns and log_df.index.name != "object_id":
            log_df_indexed = log_df.set_index("object_id")
        else:
            log_df_indexed = log_df
            
        final_df = log_df_indexed[["Z", "Z_err"]].join(features, how="right")
        
        return final_df

In [9]:
train_df = TDEFeatureEngineer().generate_features(train_log_df, train_flc_df)
test_df = TDEFeatureEngineer().generate_features(test_log_df, test_flc_df)

train_df["target"] = train_log_df["target"]

train_df

Starting Feature Engineering...
  - Deriving Peak Luminosity...
  - Deriving Rise Time...


C:\Users\nguye\AppData\Local\Temp\ipykernel_22600\2842353260.py:75: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return grouped.apply(calc_rise)


  - Deriving Power Law Decay...


C:\Users\nguye\AppData\Local\Temp\ipykernel_22600\2842353260.py:131: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return grouped.apply(calc_decay)


  - Deriving LC Smoothness...


...

  - Deriving LC Smoothness...


C:\Users\nguye\AppData\Local\Temp\ipykernel_22600\2842353260.py:188: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return grouped.apply(calc_smoothness)


  - Deriving Mean Blue Color...
  - Deriving Color Evolution...
  - Consolidating features...


C:\Users\nguye\AppData\Local\Temp\ipykernel_22600\2842353260.py:270: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return grouped.apply(calc_evolution)


,Z,Z_err,max_flux_g,max_flux_i,max_flux_r,max_flux_u,max_flux_y,max_flux_z,rise_time_u,rise_time_g,...,smoothness_u,smoothness_g,smoothness_r,smoothness_i,smoothness_z,smoothness_y,color_u_g,color_g_r,color_evol_g_r,target
object_id,,,,,,,,,,,,,,,,,,,,,
Dornhoth_anwar_melethron,1.1980,NaN,4.218222,3.116777,3.186391,5.105600,2.222071,2.880400,1393.2710,1724.5383,...,0.091390,0.105495,0.267423,0.579572,0.476415,0.235465,1.210368,1.323824,NaN,0
Dornhoth_archam_grond,0.2260,NaN,0.517468,1.528792,1.013903,1.172553,2.128189,1.854482,776.9388,9.8596,...,NaN,0.001077,0.040192,0.013512,0.132027,0.177481,2.265944,0.510372,NaN,0
Dornhoth_certh_iaun,0.4052,NaN,3.867132,7.087939,8.370757,0.596796,5.530674,7.647265,0.0000,45.1009,...,0.065518,0.257203,1.169224,0.958530,0.241498,1.375508,0.154325,0.461981,NaN,0
Dornhoth_drafn_celon,0.2748,NaN,2.306634,16.630272,16.873191,0.790957,2.648173,12.368635,49.3564,77.5600,...,0.005756,0.111295,0.940709,1.678776,0.351048,0.349017,0.342905,0.136704,NaN,0
Dornhoth_fervain_onodrim,3.0490,NaN,1.424000,28.277937,13.802700,4.275466,6.698855,29.395555,61.6134,444.4963,...,NaN,NaN,0.003486,NaN,NaN,NaN,3.002434,0.103168,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
yll_lebenedh_cair,0.3925,NaN,3.736994,7.797206,9.521811,0.863662,4.141352,7.564293,735.4949,745.4677,...,0.679285,0.080215,0.482784,0.334953,0.632553,0.860165,0.231111,0.392467,-0.311432,0
yll_merilin_rach,1.7010,NaN,1.103596,3.178706,2.366292,1.086465,5.412752,2.792981,1150.6109,1123.7633,...,NaN,NaN,0.044409,0.242199,0.445663,0.346970,0.984478,0.466382,NaN,1
yll_minai_gondrath,0.3650,NaN,0.941020,3.975947,4.434289,0.534810,3.747991,3.741118,109.9970,716.2033,...,NaN,0.202521,0.379515,0.614230,0.350210,0.097117,0.568331,0.212214,NaN,0


In [ ]:
predictor = TabularPredictor(path = f"../AutogluonModels/ag-{datetime.now().strftime("%Y%m%d-%H%M%S")}", problem_type="binary", label="target", eval_metric="f1").fit(train_df, presets = "extreme", time_limit=600)

prediction_df = pd.DataFrame({
    "object_id": test_df.index,
    "prediction": predictor.predict(test_df),
})

prediction_df

Preset alias specified: 'extreme' maps to 'extreme_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.0
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22631
CPU Count:          20
Memory Avail:       6.94 GB / 15.80 GB (44.0%)
Disk Space Avail:   22.69 GB / 823.64 GB (2.8%)
Presets specified: ['extreme']
`extreme` preset uses a dynamic portfolio based on dataset size...
	Detected data size: small (<=30000 samples), using `zeroshot_2025_tabfm` portfolio.
		Note: `zeroshot_2025_tabfm` portfolio requires a CUDA compatible GPU for best performance.
		Make sure you have all the relevant dependencies installed: `pip install autogluon.tabular[tabarena]`.
		It is strongly recommended to use a machine with 64+ GB memory and a CUDA compatible GPU with 32+ GB vRAM when using this preset. 
		This portfolio will download foundation model weights from HuggingFace during train

,object_id,prediction
object_id,,
Dornhoth_adar_imrath,Dornhoth_adar_imrath,0
Dornhoth_celeb_achad,Dornhoth_celeb_achad,0
Dornhoth_firion_fern,Dornhoth_firion_fern,0
Dornhoth_glae_aras,Dornhoth_glae_aras,0
Dornhoth_lain_tinuviel,Dornhoth_lain_tinuviel,0
...,...,...
yll_randir_bragol,yll_randir_bragol,0
yll_salph_meril,yll_salph_meril,0
yll_sogannen_lain,yll_sogannen_lain,0


In [ ]:
Path("../output").mkdir(parents=True, exist_ok=True)

prediction_df.to_csv(f"../output/submission-{datetime.now().strftime("%Y%m%d-%H%M%S")}.csv", index=False)

In [13]:
# class AbstractFeatureExtractor:

# class PhysicalFeatureExtractor:
#     """
#     Extracts physically motivated features for TDE classification from light curves
#     and metadata.
#     """

#     def __init__(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> None:
#         self.__log_df = log_df
        
#         self.__flc_df: pd.DataFrame = self.__apply_extinction_correction(flc_df)
        
#         # Cache for expensive regression results to avoid re-computing for multiple features
#         self.__decay_cache: Optional[pd.DataFrame] = None

#     def extract(self) -> pd.DataFrame:
#         """
#         Main execution pipeline. Aggregates all derived features into a single DataFrame.
#         """
#         # Pre-compute expensive decay stats to populate the cache
#         self.__decay_cache = self.__compute_decay_statistics()

#         return self.__log_df.join([
#             self.__extract_mean_blue_color(),
#             self.__extract_color_evolution(),
#             self.__extract_power_law_decay(),
#             self.__extract_peak_luminosity(),
#             self.__extract_lc_smoothness(),
#         ])

#     def __apply_extinction_correction(self, log_df: pd.DataFrame, flc_df: pd.DataFrame) -> pd.DataFrame:
#         """
#         Corrects Flux and Flux_err for Galactic Extinction using E(B-V).
#         Formula: F_corr = F_obs * 10^(0.4 * A_lambda)
#         Approximation: A_lambda approx R_V * E(B-V) with R_V ~ 3.1
#         """
#         return (
#             flc_df
#             .merge(log_df[["EBV"]], left_on="object_id", right_index=True, how="left")
#             .assign(
#                 correction_factor=lambda x: np.pow(10, 0.4 * 3.1 * x["EBV"]),
#                 Flux_corr=lambda x: x["Flux"] * x["correction_factor"],
#                 Flux_err_corr=lambda x: x["Flux_err"] * x["correction_factor"]
#             )
#             .drop(columns=["EBV", "correction_factor"])
#         )

#     def __extract_mean_blue_color(self) -> pd.Series:
#         """
#         Derives mean 'u - g' color proxy.
#         Formula: -2.5 * log10( Mean_Flux_u / Mean_Flux_g )
#         [cite_start]Theory: TDEs are hot (Blue), so Flux_u should be high relative to Flux_g[cite: 17, 19].
#         """
#         # Calculate mean flux per object per filter
#         pivot_df = (
#             self.__flc_df
#             .groupby(["object_id", "Filter"])["Flux_corr"]
#             .mean()
#             .unstack()
#         )

#         # Calculate Color: -2.5 * log10(u / g)
#         # Note: We use .get() to handle cases where 'u' or 'g' might be missing entirely
#         u_flux = pivot_df.get("u", pd.Series(np.nan, index=pivot_df.index))
#         g_flux = pivot_df.get("g", pd.Series(np.nan, index=pivot_df.index))

#         color_series = -2.5 * np.log10(
#             (u_flux + EPS) / (g_flux + EPS)
#         )
        
#         return color_series.rename("mean_blue_color")

#     def __extract_color_evolution(self) -> pd.Series:
#         """
#         Derives the rate of color change.
#         Formula: Slope of (Flux_g / Flux_r) over time.
#         Theory: TDEs have constant Temperature (slope ~ 0). [cite_start]SNe cool/redden (slope < 0)[cite: 18].
#         """
#         def calculate_slope(group: pd.DataFrame) -> float:
#             # Pivot to get time-aligned g and r bands
#             pivoted = group.pivot_table(
#                 index="Time (MJD)", 
#                 columns="Filter", 
#                 values="Flux_corr", 
#                 aggfunc="mean"
#             )
            
#             if "g" not in pivoted.columns or "r" not in pivoted.columns:
#                 return np.nan
            
#             # Calculate ratio g/r
#             ratio = pivoted["g"] / (pivoted["r"] + EPS)
#             valid_data = ratio.dropna()
            
#             # Need at least 3 points for a meaningful regression
#             if len(valid_data) < 3:
#                 return np.nan
            
#             slope, _, _, _, _ = linregress(valid_data.index, valid_data.values)
#             return slope

#         # Filter for relevant bands and apply calculation
#         return (
#             self.__flc_df[self.__flc_df["Filter"].isin(["g", "r"])]
#             .groupby("object_id")
#             .apply(calculate_slope)
#             .rename("color_evolution")
#         )

#     def __compute_decay_statistics(self) -> pd.DataFrame:
#         """
#         Helper method: Fits Power Law to light curves to derive Decay Alpha and Smoothness.
#         Formula: log(Flux) = C - alpha * log(t - t_peak)
#         [cite_start]Theory: TDEs follow t^-5/3[cite: 18, 1294].
#         """
#         def fit_decay(group: pd.DataFrame) -> pd.Series:
#             # Use 'g' band if available, else all data
#             g_band = group[group["Filter"] == "g"]
#             fit_data = g_band if not g_band.empty else group
            
#             if fit_data.empty:
#                 return pd.Series({"alpha": np.nan, "mse": np.nan})

#             # Identify Peak
#             try:
#                 peak_idx = fit_data["Flux_corr"].idxmax()
#                 t_peak = fit_data.loc[peak_idx, "Time (MJD)"]
#             except ValueError:
#                 return pd.Series({"alpha": np.nan, "mse": np.nan})

#             # Select Post-Peak Data (> 5 days after peak to avoid peak turnover)
#             # Must be positive for log-space fitting
#             mask = (fit_data["Time (MJD)"] > (t_peak + 5)) & (fit_data["Flux_corr"] > 0)
#             post_peak = fit_data[mask]

#             if len(post_peak) < 5:
#                 return pd.Series({"alpha": np.nan, "mse": np.nan})

#             # Prepare Log-Log data
#             X = np.log10(post_peak["Time (MJD)"] - t_peak + EPS)
#             Y = np.log10(post_peak["Flux_corr"])

#             # Linear Regression
#             slope, intercept, _, _, _ = linregress(X, Y)
            
#             # Calculate MSE (Smoothness proxy)
#             y_pred = slope * X + intercept
#             mse = np.mean((Y - y_pred) ** 2)

#             # Return Alpha (negative of slope) and MSE
#             return pd.Series({"alpha": -slope, "mse": mse})

#         return (
#             self.__flc_df
#             .groupby("object_id")
#             .apply(fit_decay)
#         )

#     def __extract_power_law_decay(self) -> pd.Series:
#         """
#         Returns the power-law decay index (alpha).
#         """
#         if self.__decay_cache is None:
#             self.__decay_cache = self.__compute_decay_statistics()
            
#         return self.__decay_cache["alpha"].rename("power_law_decay")

#     def __extract_peak_luminosity(self) -> pd.Series:
#         """
#         Derives a proxy for Peak Absolute Magnitude/Luminosity.
#         Formula: log10(Flux_peak) + 2 * log10(Redshift)
#         [cite_start]Theory: L propto F * d^2. d propto z[cite: 15].
#         """
#         # Calculate max flux per object
#         max_flux = self.__flc_df.groupby("object_id")["Flux_corr"].max()
        
#         # Align with metadata redshift
#         # Use pandas alignment (index matching) automatically
#         z = self.__log_df["Z"]
        
#         lum_proxy = np.log10(max_flux + EPS) + 2 * np.log10(z + EPS)
        
#         return lum_proxy.rename("peak_luminosity")

#     def __extract_lc_smoothness(self) -> pd.Series:
#         """
#         Returns the smoothness metric (MSE of power-law fit).
#         Theory: Low MSE = Smooth (TDE). [cite_start]High MSE = Stochastic (AGN)[cite: 22].
#         """
#         if self.__decay_cache is None:
#             self.__decay_cache = self.__compute_decay_statistics()
            
#         return self.__decay_cache["mse"].rename("lc_smoothness")